In [1]:
# =========================================================
# NLP + K-MEANS CLUSTERING
# File: hello-world-stock.csv
# =========================================================

# 1. Install libraries (run once)
!pip install pandas numpy scikit-learn matplotlib nltk -q

# 2. Import libraries
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

nltk.download("stopwords", quiet=True)

# 3. Load dataset
df = pd.read_csv("hello-world-stock.csv")

print("Shape:", df.shape)
print(df.head())
print("\nColumns:", df.columns.tolist())

# 4. Select text column automatically
text_cols = df.select_dtypes(include="object").columns

if len(text_cols) == 0:
    raise ValueError("No text column found in the CSV.")

names = ["text", "headline", "title", "news", "description", "content"]

text_col = next(
    (c for c in text_cols if any(n in c.lower() for n in names)),
    text_cols[0]
)

print("\nUsing text column:", text_col)

# 5. Clean text
stop_words = set(stopwords.words("english"))

def clean(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

df["clean_text"] = df[text_col].fillna("").apply(clean)

# Remove empty records
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

# 6. TF-IDF
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(df["clean_text"])

print("\nTF-IDF shape:", X.shape)

# 7. Find best K using silhouette score
scores = {}

for k in range(2, min(10, len(df) - 1) + 1):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)

print("\nBest K:", best_k)
print("Silhouette score:", round(scores[best_k], 4))

# 8. Final K-Means
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

# 9. Show cluster keywords
terms = tfidf.get_feature_names_out()

for i in range(best_k):
    top = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(terms[top]))

# 10. PCA visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    cmap="viridis"
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means NLP Clusters")
plt.colorbar(label="Cluster")
plt.show()

# 11. Show results
print("\nCluster counts:")
print(df["cluster"].value_counts().sort_index())

display(df[[text_col, "cluster"]].head(20))

# 12. Save results
df.to_csv(
    "hello-world-stock-kmeans-results.csv",
    index=False
)

print("\nSaved as: hello-world-stock-kmeans-results.csv")

Shape: (2261, 15)
   Unnamed: 0        Date    Open     High      Low   Close      Volume  \
0           0  2017-12-29  170.52  170.590  169.220  169.23  25643711.0   
1           1  2017-12-28  171.00  171.850  170.480  171.08  15997739.0   
2           2  2017-12-27  170.10  170.780  169.710  170.60  21672062.0   
3           3  2017-12-26  170.80  171.470  169.679  170.57  32968167.0   
4           4  2017-12-22  174.68  175.424  174.500  175.01  16052615.0   

   ExDividend  SplitRatio  AdjOpen  AdjHigh   AdjLow  AdjClose   AdjVolume  \
0         0.0         1.0   170.52  170.590  169.220    169.23  25643711.0   
1         0.0         1.0   171.00  171.850  170.480    171.08  15997739.0   
2         0.0         1.0   170.10  170.780  169.710    170.60  21672062.0   
3         0.0         1.0   170.80  171.470  169.679    170.57  32968167.0   
4         0.0         1.0   174.68  175.424  174.500    175.01  16052615.0   

  Stock  
0  AAPL  
1  AAPL  
2  AAPL  
3  AAPL  
4  AAPL  

C

/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_4407/3093955750.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


ValueError: empty vocabulary; perhaps the documents only contain stop words